# Readers

Notebook was prepared as a sample tool to read data from different data sources. Moreover notebook includes some simple exploratory data analysis for data overview purposes.

In [1]:
# Reload modules to reflect changes made in external scripts
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
from pathlib import Path
from pprint import pprint as print

import requests
import pandas as pd
import numpy as np

In [3]:
CURRENT_WORKING_DIR = Path(os.getcwd())
WORKING_DIR = CURRENT_WORKING_DIR.parent
DATA_DIR = WORKING_DIR / "data"

sys.path.append(str(WORKING_DIR))

## SkillCorner

Data loading is based on the SkillCorner tutorial guide. More information under below link :arrow_down:

**Link:** https://github.com/SkillCorner/opendata

In [4]:
# Content URL for the SkillCorner opendata repository
content_url = "https://api.github.com/repos/SkillCorner/opendata/contents/data/matches"

# Get the list of available matches
response = requests.get(content_url)
matches = response.json()

# Display available match IDs
match_ids = [match['name'] for match in matches if match['type'] == 'dir']
print(f"Available matches: {len(match_ids)}")
print(match_ids)

'Available matches: 10'
['1886347',
 '1899585',
 '1925299',
 '1953632',
 '1996435',
 '2006229',
 '2011166',
 '2013725',
 '2015213',
 '2017461']


## Data Readers ###

In [5]:
# Example: Load tracking data for a specific match
match_id = 1886347

# Construct the raw tracking content URL
content_tracking_url = content_url + f"/{match_id}/{match_id}_tracking_extrapolated.jsonl"  # Data is stored using GitLFS
content_tracking_data = pd.read_json(content_tracking_url, lines=True)
print(f"Downloaded tracking data URL: {content_tracking_data['download_url'][0]}")

# Construct the raw match content URL
content_match_url = content_url + f"/{match_id}/{match_id}_match.json"  # Data is stored using GitLFS
content_match_data = pd.read_json(content_match_url, lines=True)
print(f"Downloaded match data URL: {content_match_data['download_url'][0]}")

('Downloaded tracking data URL: '
 'https://media.githubusercontent.com/media/SkillCorner/opendata/master/data/matches/1886347/1886347_tracking_extrapolated.jsonl')
('Downloaded match data URL: '
 'https://raw.githubusercontent.com/SkillCorner/opendata/master/data/matches/1886347/1886347_match.json')


## Players & Ball tracking data

Extract player and ball data. Preprocess raw data frame into processed, ready for analytics data frame. 

In [6]:
from football_ml.preprocessors.tracking import (
    preprocess_player_tracking,
    preprocess_ball_tracking
)

# Read the JSON tracking data as a JSON object
raw_tracking_data = pd.read_json(content_tracking_data['download_url'][0], lines=True)

# Preprocess player tracking data
player_tracking_df = preprocess_player_tracking(raw_tracking_data, match_id=match_id)
print(f"Player Tracking DataFrame shape: {player_tracking_df.shape}")

# Preprocess ball tracking data
ball_tracking_df = preprocess_ball_tracking(raw_tracking_data, match_id=match_id)
print(f"Ball Tracking DataFrame shape: {ball_tracking_df.shape}")

'Player Tracking DataFrame shape: (956076, 8)'
'Ball Tracking DataFrame shape: (43458, 10)'


In [7]:
player_tracking_df.head()

,player_tracking_id,timestamp,player_x,player_y,period,frame,player_id,match_id
0,0,2026-01-04,-39.63,-0.08,1.0,10,51009,1886347
1,1,2026-01-04,-19.21,-9.18,1.0,10,176224,1886347
2,2,2026-01-04,-21.83,0.47,1.0,10,51649,1886347
3,3,2026-01-04,-1.16,-32.47,1.0,10,50983,1886347
4,4,2026-01-04,-18.88,15.73,1.0,10,735578,1886347


In [8]:
ball_tracking_df.head()

,ball_tracking_id,timestamp,ball_x,ball_y,ball_z,is_detected,period,match_id,possession_player_id,possession_team_group
0,0,2026-01-04 00:00:00.000,0.32,0.38,0.13,True,1.0,1886347,NaN,None
1,1,2026-01-04 00:00:00.100,0.54,0.08,0.22,True,1.0,1886347,NaN,None
2,2,2026-01-04 00:00:00.200,0.57,-0.07,0.19,True,1.0,1886347,NaN,None
3,3,2026-01-04 00:00:00.300,0.56,-0.07,0.14,True,1.0,1886347,NaN,None
4,4,2026-01-04 00:00:00.400,0.59,-0.03,0.14,True,1.0,1886347,NaN,None


## Players & Match facts data

In [9]:
from football_ml.preprocessors.match import (
    preprocess_competition_season_data,
    preprocess_match_data,
    preprocess_teams_data,
    preprocess_match_players_data,
    preprocess_players_data,
    preprocess_stadium_data,
    preprocess_player_position_data,
)

# Read the JSON match data as a JSON object
response = requests.get(content_match_data['download_url'][0])
raw_match_data = response.json()
raw_match_df = pd.json_normalize(raw_match_data, max_level=2)

In [11]:
competition_season_df = preprocess_competition_season_data(raw_match_df)
print(f"Competition Season DataFrame shape: {competition_season_df.shape}")

match_df = preprocess_match_data(raw_match_df)
print(f"Match DataFrame shape: {match_df.shape}")

teams_df = preprocess_teams_data(raw_match_df)
print(f"Teams DataFrame shape: {teams_df.shape}")

match_players_df = preprocess_match_players_data(raw_match_df)
print(f"Match Players DataFrame shape: {match_players_df.shape}")

players_df = preprocess_players_data(raw_match_df)
print(f"Players DataFrame shape: {players_df.shape}")

stadium_df = preprocess_stadium_data(raw_match_df)
print(f"Stadium DataFrame shape: {stadium_df.shape}")

player_position_df = preprocess_player_position_data(raw_match_df)
print(f"Player Position DataFrame shape: {player_position_df.shape}")

'Competition Season DataFrame shape: (1, 6)'
'Match DataFrame shape: (1, 16)'
'Teams DataFrame shape: (2, 4)'
'Match Players DataFrame shape: (36, 19)'
'Players DataFrame shape: (36, 12)'
'Stadium DataFrame shape: (1, 7)'
'Player Position DataFrame shape: (36, 4)'


# Read preprocessed data from parquet files 

In [12]:
competition_season_df = pd.read_parquet(DATA_DIR / "raw" / "dim_competition_season.parquet")
match_df = pd.read_parquet(DATA_DIR / "raw" / "dim_match.parquet")
teams_df = pd.read_parquet(DATA_DIR / "raw" / "dim_team.parquet")
players_df = pd.read_parquet(DATA_DIR / "raw" / "dim_players.parquet")
stadium_df = pd.read_parquet(DATA_DIR / "raw" / "dim_stadium.parquet")
match_players_df = pd.read_parquet(DATA_DIR / "raw" / "fct_match_players.parquet")
player_position_df = pd.read_parquet(DATA_DIR / "raw" / "dim_players_position.parquet")
ball_tracking_df = pd.read_parquet(DATA_DIR / "raw" / "fct_ball_tracking.parquet")
player_tracking_df = pd.read_parquet(DATA_DIR / "raw" / "fct_players_tracking.parquet")   

**Analysis ideas:**
 - free kicks, corners - players movements
 - Midfielders - off ball movements, number of offensive / defensive passes, passes based on the number of opponent players
 - Center Backs - off ball movements in offensive part of the game, passes
 - Before goal team movement

**ML model ideas:**
- Clustering - players similarities
    - off ball movements